## **LLM Benchmarking**

### **Data Preparation**

Pull test data and class names from artifacts

In [1]:
import joblib

x_test = joblib.load(r".\Artifacts\x_test.pkl")
y_test = joblib.load(r".\Artifacts\y_test.pkl")
class_names = joblib.load(r".\Artifacts\class_names.pkl")

print("Data split loaded")

Data split loaded


In [27]:
x_test[:10]

['today tomorrow try move mountains cross fingers',
 'typa girl bullied lesbians high school',
 'cool woman gave directions polling place whose wee boy always runs around high heels ‘girls’ clothes he’s told gets bullied ‘poof’',
 'need teach version quran terrorists killing people world blast selves kill innocent isis terror org bearing enough bull shits cant buy version see jehad gazwa hind darulharb',
 'rome destroy coliseum roman forum every building statue romans slaves persecutedkilled christians support radical “muslims” iraq syria blew historic cultural sites',
 'try hide jihadi propaganda know hell gys specly jihadin aunty urgnt need hide islamic terrorisms behind typ artificial propaganda coz current scenario exposed infrnt whole world',
 'city college',
 'pepperidge farm remembers',
 'added feminazi white knight professional victims terminology',
 'blm something know muslim leaders record stating use manipulate goals quoted saying islam’s ‘useful idiots’ islam says that whit

In [24]:
y_test[:10]

array([4, 0, 0, 5, 5, 5, 3, 4, 4, 5])

In [28]:
len(x_test), len(y_test)

(11923, 11923)

In [4]:
class_names

Index(['age', 'ethnicity', 'gender', 'not_cyberbullying',
       'other_cyberbullying', 'religion'],
      dtype='object')

### **LLM: Qwen2.5:7b-Instruct**

#### **Predictions**

In [15]:
import os
import json
import re
import time
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import joblib

# ----------------------------
# Ollama configuration
# ----------------------------
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5:7b-instruct")  # e.g., deepseek-r1:7b, qwen2.5:7b-instruct
MAX_OLLAMA_CASES = None  # set integer (e.g., 200) for a smaller quick run
REQUEST_TIMEOUT_SECONDS = 120
MAX_RETRIES = 3

# Checkpoint/resume configuration
CHECKPOINT_EVERY = 25
RESUME_FROM_CHECKPOINT = True
RESET_CHECKPOINT = False

# If earlier cells were not run, load artifacts here as a fallback.
if "x_test" not in globals():
    x_test = joblib.load(r".\Artifacts\x_test.pkl")
if "y_test" not in globals():
    y_test = joblib.load(r".\Artifacts\y_test.pkl")
if "class_names" not in globals():
    class_names = joblib.load(r".\Artifacts\class_names.pkl")

class_names_list = list(class_names)
label_to_id = {label: i for i, label in enumerate(class_names_list)}
FALLBACK_LABEL = "other_cyberbullying"

if MAX_OLLAMA_CASES is None:
    x_eval = list(x_test)
    y_eval = np.asarray(y_test)
else:
    x_eval = list(x_test)[:MAX_OLLAMA_CASES]
    y_eval = np.asarray(y_test)[:MAX_OLLAMA_CASES]

artifacts_dir = Path(r".\Artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
safe_model_name = re.sub(r"[^a-zA-Z0-9_.-]", "_", OLLAMA_MODEL)
checkpoint_path = artifacts_dir / f"ollama_checkpoint_{safe_model_name}.pkl"
predictions_path = artifacts_dir / f"ollama_predictions_{safe_model_name}.pkl"

if RESET_CHECKPOINT and checkpoint_path.exists():
    checkpoint_path.unlink()
    print(f"Deleted old checkpoint: {checkpoint_path}")

SYSTEM_PROMPT = f"""You are an expert annotator for cyberbullying tweet classification.

Valid labels: {class_names_list}

Rules:
1) Return exactly one label from the valid label list.
2) Output STRICT JSON only, with this schema:
   {{"label": "<one_valid_label>"}}
3) Do not output markdown, code fences, or any extra keys.
"""

def _parse_json_payload(raw_text: str):
    text = (raw_text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.IGNORECASE | re.DOTALL).strip()
    return json.loads(text)

def _normalize_text(s: str) -> str:
    return re.sub(r"[^a-z0-9_]+", "", (s or "").strip().lower())

normalized_label_to_canonical = {
    _normalize_text(label): label for label in class_names_list
}

def _map_to_valid_label(raw_label: str) -> str:
    # 1) Exact canonical label
    if raw_label in label_to_id:
        return raw_label

    raw_norm = _normalize_text(raw_label)

    # 2) Exact normalized match (handles spaces/case/punctuation variants)
    if raw_norm in normalized_label_to_canonical:
        return normalized_label_to_canonical[raw_norm]

    # 3) Substring match against any class label
    for cls in class_names_list:
        cls_norm = _normalize_text(cls)
        if cls_norm and cls_norm in raw_norm:
            return cls

    # 4) Fallback if no class detected
    if FALLBACK_LABEL in label_to_id:
        return FALLBACK_LABEL

    raise ValueError(
        f"Invalid label returned and fallback label not found: {raw_label}. "
        f"Expected one of {class_names_list} with fallback '{FALLBACK_LABEL}'."
    )

def ollama_predict_one(tweet_text: str):
    last_error = None
    payload = {
        "model": OLLAMA_MODEL,
        "stream": False,
        "format": "json",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Classify this tweet:\n\n{tweet_text}"},
        ],
        "options": {"temperature": 0},
    }

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(
                f"{OLLAMA_BASE_URL}/api/chat",
                json=payload,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            resp.raise_for_status()
            data = resp.json()
            content = data.get("message", {}).get("content", "")
            out = _parse_json_payload(content)
            raw_label = str(out.get("label", "")).strip()
            final_label = _map_to_valid_label(raw_label)
            return label_to_id[final_label], final_label
        except Exception as err:
            last_error = err
            if attempt < MAX_RETRIES:
                time.sleep(1.2 * attempt)

    raise RuntimeError(f"Ollama prediction failed after retries: {last_error}")

def save_checkpoint(next_index: int, pred_ids, pred_rows):
    state = {
        "model": OLLAMA_MODEL,
        "x_eval_size": len(x_eval),
        "next_index": int(next_index),
        "pred_ids": list(pred_ids),
        "pred_rows": list(pred_rows),
        "saved_at_unix": time.time(),
    }
    joblib.dump(state, checkpoint_path)

# ----------------------------
# Resume from checkpoint if available
# ----------------------------
ollama_pred_ids = []
ollama_rows = []
start_index = 0

if RESUME_FROM_CHECKPOINT and checkpoint_path.exists():
    state = joblib.load(checkpoint_path)
    same_model = state.get("model") == OLLAMA_MODEL
    same_size = int(state.get("x_eval_size", -1)) == len(x_eval)
    if same_model and same_size:
        ollama_pred_ids = list(state.get("pred_ids", []))
        ollama_rows = list(state.get("pred_rows", []))
        start_index = int(state.get("next_index", len(ollama_pred_ids)))
        start_index = max(0, min(start_index, len(x_eval)))
        print(f"Resumed from checkpoint: {checkpoint_path}")
        print(f"Loaded {len(ollama_pred_ids)} saved predictions; continuing from index {start_index}")
    else:
        print("Checkpoint found but skipped due to model/test-size mismatch.")
        print(f"Checkpoint path: {checkpoint_path}")

# ----------------------------
# Run Ollama prediction for each tweet
# ----------------------------
total = len(x_eval)
start_time = time.time()

for idx in range(start_index, total):
    tweet_text = x_eval[idx]
    pred_id, pred_label = ollama_predict_one(str(tweet_text))
    ollama_pred_ids.append(int(pred_id))
    ollama_rows.append({
        "tweet": str(tweet_text),
        "y_test_label": class_names_list[int(y_eval[idx])],
        "ollama_prediction": pred_label,
    })

    processed = idx + 1
    if (processed % CHECKPOINT_EVERY == 0) or (processed == total):
        save_checkpoint(next_index=processed, pred_ids=ollama_pred_ids, pred_rows=ollama_rows)
        elapsed = time.time() - start_time
        done_now = max(1, processed - start_index)
        sec_per_tweet = elapsed / done_now
        remaining = total - processed
        eta_min = (remaining * sec_per_tweet) / 60.0
        print(f"Ollama progress: {processed}/{total} | checkpoint saved | ETA ~ {eta_min:.1f} min")

# Convert and save final artifacts
ollama_pred_ids = np.asarray(ollama_pred_ids, dtype=int)
ollama_pred_df = pd.DataFrame(ollama_rows)
ollama_pred_df["is_match"] = ollama_pred_df["y_test_label"] == ollama_pred_df["ollama_prediction"]

prediction_compare_df = ollama_pred_df[[
    "tweet",
    "y_test_label",
    "ollama_prediction",
    "is_match",
]].copy()

joblib.dump(
    {
        "ollama_pred_ids": ollama_pred_ids,
        "prediction_compare_df": prediction_compare_df,
        "model": OLLAMA_MODEL,
        "x_eval_size": len(x_eval),
    },
    predictions_path,
 )

display(prediction_compare_df.head(20))
print(f"Done. Collected {len(ollama_pred_ids)} predictions from model: {OLLAMA_MODEL}")
print(f"Total exact label matches: {int(prediction_compare_df['is_match'].sum())}/{len(prediction_compare_df)}")
print(f"Checkpoint file: {checkpoint_path}")
print(f"Predictions artifact: {predictions_path}")

Ollama progress: 25/11923 | checkpoint saved | ETA ~ 1063.2 min
Ollama progress: 50/11923 | checkpoint saved | ETA ~ 1083.9 min
Ollama progress: 75/11923 | checkpoint saved | ETA ~ 1075.7 min
Ollama progress: 100/11923 | checkpoint saved | ETA ~ 1102.7 min
Ollama progress: 125/11923 | checkpoint saved | ETA ~ 1120.7 min
Ollama progress: 150/11923 | checkpoint saved | ETA ~ 1130.2 min
Ollama progress: 175/11923 | checkpoint saved | ETA ~ 1142.3 min
Ollama progress: 200/11923 | checkpoint saved | ETA ~ 1140.6 min
Ollama progress: 225/11923 | checkpoint saved | ETA ~ 1143.3 min
Ollama progress: 250/11923 | checkpoint saved | ETA ~ 1150.0 min
Ollama progress: 275/11923 | checkpoint saved | ETA ~ 1156.7 min
Ollama progress: 300/11923 | checkpoint saved | ETA ~ 1155.2 min
Ollama progress: 325/11923 | checkpoint saved | ETA ~ 1146.6 min
Ollama progress: 350/11923 | checkpoint saved | ETA ~ 1140.9 min
Ollama progress: 375/11923 | checkpoint saved | ETA ~ 1133.7 min
Ollama progress: 400/11923 |

,tweet,y_test_label,ollama_prediction,is_match
0,today tomorrow try move mountains cross fingers,other_cyberbullying,not_cyberbullying,False
1,typa girl bullied lesbians high school,age,gender,False
2,cool woman gave directions polling place whose...,age,gender,False
3,need teach version quran terrorists killing pe...,religion,religion,True
4,rome destroy coliseum roman forum every buildi...,religion,religion,True
5,try hide jihadi propaganda know hell gys specl...,religion,religion,True
6,city college,not_cyberbullying,not_cyberbullying,True
7,pepperidge farm remembers,other_cyberbullying,not_cyberbullying,False
8,added feminazi white knight professional victi...,other_cyberbullying,gender,False
9,blm something know muslim leaders record stati...,religion,religion,True


Done. Collected 11923 predictions from model: qwen2.5:7b-instruct
Total exact label matches: 5277/11923
Checkpoint file: Artifacts\ollama_checkpoint_qwen2.5_7b-instruct.pkl
Predictions artifact: Artifacts\ollama_predictions_qwen2.5_7b-instruct.pkl


#### **Evaluation**

In [18]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# Ensure y_eval is integer index array and lengths match predictions.
y_eval_idx = np.asarray(y_eval).astype(int)
ollama_pred_idx = np.asarray(ollama_pred_ids).astype(int)

if len(y_eval_idx) != len(ollama_pred_idx):
    raise ValueError(
        f"Length mismatch: y_eval has {len(y_eval_idx)} rows, ollama_pred has {len(ollama_pred_idx)} rows"
    )

# Keep readable labels and index columns for inspection.
ollama_pred_df["true_label_idx"] = y_eval_idx
ollama_pred_df["ollama_pred_idx"] = ollama_pred_idx
display(ollama_pred_df.head(10))

p, r, f1, support = precision_recall_fscore_support(
    y_eval_idx,
    ollama_pred_idx,
    average="weighted",
    zero_division=0,
 )

ollama_metrics = {
    "Accuracy": accuracy_score(y_eval_idx, ollama_pred_idx),
    "Precision": p,
    "Recall": r,
    "F1-Score": f1,
    "Support": int(len(y_eval_idx)),
}

ollama_metrics_df = pd.DataFrame(ollama_metrics, index=[OLLAMA_MODEL]).T
display(ollama_metrics_df)

print("\nLLM classification report:\n")
print(
    classification_report(
        y_eval_idx,
        ollama_pred_idx,
        target_names=class_names_list,
        zero_division=0,
    )
)

,tweet,y_test_label,ollama_prediction,is_match,true_label_idx,ollama_pred_idx
0,today tomorrow try move mountains cross fingers,other_cyberbullying,not_cyberbullying,False,4,3
1,typa girl bullied lesbians high school,age,gender,False,0,2
2,cool woman gave directions polling place whose...,age,gender,False,0,2
3,need teach version quran terrorists killing pe...,religion,religion,True,5,5
4,rome destroy coliseum roman forum every buildi...,religion,religion,True,5,5
5,try hide jihadi propaganda know hell gys specl...,religion,religion,True,5,5
6,city college,not_cyberbullying,not_cyberbullying,True,3,3
7,pepperidge farm remembers,other_cyberbullying,not_cyberbullying,False,4,3
8,added feminazi white knight professional victi...,other_cyberbullying,gender,False,4,2
9,blm something know muslim leaders record stati...,religion,religion,True,5,5


,qwen2.5:7b-instruct
Accuracy,0.442590
Precision,0.524460
Recall,0.442590
F1-Score,0.428401
Support,11923.000000



LLM classification report:

                     precision    recall  f1-score   support

                age       0.51      0.02      0.04      1952
          ethnicity       0.79      0.42      0.55      1995
             gender       0.48      0.44      0.46      1952
  not_cyberbullying       0.39      0.69      0.50      2040
other_cyberbullying       0.13      0.23      0.17      2006
           religion       0.85      0.85      0.85      1978

           accuracy                           0.44     11923
          macro avg       0.53      0.44      0.43     11923
       weighted avg       0.52      0.44      0.43     11923



### **Zero-shot Classifier Models**

#### **Predictions**

In [ ]:
# Additional zero-shot benchmark across 3 models (BART + ModernBERT + DeBERTa-v3)
import re
import time
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from transformers import pipeline
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

try:
    from datasets import Dataset
    from transformers.pipelines.pt_utils import KeyDataset
except ImportError as e:
    raise ImportError(
        "datasets is required for efficient GPU batching. Install with: pip install datasets"
    ) from e

# ---------------------------------------------------------
# Models to benchmark (edit these IDs if you want variants)
# ---------------------------------------------------------
ZS_MODELS = [
    "facebook/bart-large-mnli",
    "MoritzLaurer/ModernBERT-large-zeroshot-v2.0",
    "MoritzLaurer/deberta-v3-large-zeroshot-v2.0",
]

# Set to None to use all tweets. Set an int (e.g., 1000) for a quick sample run.
MAX_ZS_CASES = None

ZS_BATCH_SIZE = 16
ZS_HYPOTHESIS_TEMPLATE = "This tweet belongs to the {} category."
ZS_CHECKPOINT_EVERY = 1000
ZS_RESUME_FROM_CHECKPOINT = True
ZS_RESET_CHECKPOINT = False

# Use CUDA when available; otherwise use CPU.
ZS_DEVICE = 0 if torch.cuda.is_available() else -1
print(f"Multi-model zero-shot device: {'cuda' if ZS_DEVICE == 0 else 'cpu'}")

# Reuse prepared split when available.
if "x_eval" not in globals() or "y_eval" not in globals():
    x_eval = list(x_test)
    y_eval = np.asarray(y_test)

if "class_names_list" not in globals():
    class_names_list = [str(x) for x in list(class_names)]

candidate_labels = list(class_names_list)
label_to_id = {label: i for i, label in enumerate(candidate_labels)}
if len(candidate_labels) == 0:
    raise ValueError("candidate_labels is empty. Please ensure class_names is loaded.")

# Reuse existing artifacts directory if set; otherwise default to ./Artifacts.
if "artifacts_dir" in globals():
    artifacts_dir_multi = Path(artifacts_dir)
else:
    artifacts_dir_multi = Path(r"./Artifacts")
artifacts_dir_multi.mkdir(parents=True, exist_ok=True)

def _sanitize_text(t) -> str:
    text = "" if t is None else str(t)
    text = text.strip()
    return text if text else "[EMPTY_TWEET]"

# Optional quick subset for all 3 models.
if MAX_ZS_CASES is None:
    x_eval_run = list(x_eval)
    y_eval_run = np.asarray(y_eval)
else:
    x_eval_run = list(x_eval)[: int(MAX_ZS_CASES)]
    y_eval_run = np.asarray(y_eval)[: int(MAX_ZS_CASES)]

x_eval_text = [_sanitize_text(t) for t in x_eval_run]
y_eval_idx = np.asarray(y_eval_run).astype(int)
total = len(x_eval_text)
if total == 0:
    raise ValueError("x_eval has no samples to predict.")

print(f"Running multi-model zero-shot on {total} tweets (MAX_ZS_CASES={MAX_ZS_CASES})")

def _run_zeroshot_model(model_name: str):
    safe_name = re.sub(r"[^a-zA-Z0-9_.-]", "_", model_name)
    size_tag = f"_{total}"
    checkpoint_path = artifacts_dir_multi / f"nli_checkpoint_{safe_name}{size_tag}.pkl"
    predictions_path = artifacts_dir_multi / f"nli_predictions_{safe_name}{size_tag}.pkl"

    if ZS_RESET_CHECKPOINT and checkpoint_path.exists():
        checkpoint_path.unlink()
        print(f"Deleted old checkpoint: {checkpoint_path}")

    pred_ids_list = []
    pred_labels = []
    pred_rows = []
    start_index = 0

    if ZS_RESUME_FROM_CHECKPOINT and checkpoint_path.exists():
        state = joblib.load(checkpoint_path)
        same_model = state.get("model") == model_name
        same_size = int(state.get("x_eval_size", -1)) == total
        same_labels = state.get("candidate_labels", []) == candidate_labels
        if same_model and same_size and same_labels:
            pred_ids_list = list(state.get("pred_ids", []))
            pred_labels = list(state.get("pred_labels", []))
            pred_rows = list(state.get("pred_rows", []))
            start_index = int(state.get("next_index", len(pred_labels)))
            start_index = max(0, min(start_index, total))
            print(f"Resumed checkpoint for {model_name}: {len(pred_labels)} predictions")
        else:
            print(f"Skipped mismatched checkpoint for {model_name}")

    def _save_ckpt(next_index: int, pred_ids, labels, rows):
        state = {
            "model": model_name,
            "candidate_labels": candidate_labels,
            "x_eval_size": total,
            "next_index": int(next_index),
            "pred_ids": list(pred_ids),
            "pred_labels": list(labels),
            "pred_rows": list(rows),
            "saved_at_unix": time.time(),
        }
        joblib.dump(state, checkpoint_path)

    clf = pipeline(
        task="zero-shot-classification",
        model=model_name,
        device=ZS_DEVICE,
    )

    if start_index < total:
        run_dataset = Dataset.from_dict({"text": x_eval_text[start_index:]})
        keyed = KeyDataset(run_dataset, "text")
        t0 = time.time()
        for local_i, out in enumerate(
            clf(
                keyed,
                candidate_labels=candidate_labels,
                multi_label=False,
                hypothesis_template=ZS_HYPOTHESIS_TEMPLATE,
                batch_size=ZS_BATCH_SIZE,
                truncation=True,
            ),
            start=1,
        ):
            pred_label = out["labels"][0]
            pred_id = int(label_to_id[pred_label])
            idx = start_index + local_i - 1

            pred_labels.append(pred_label)
            pred_ids_list.append(pred_id)
            pred_rows.append({
                "tweet": x_eval_text[idx],
                "y_test_label": candidate_labels[int(y_eval_idx[idx])],
                "nli_prediction": pred_label,
            })

            processed = start_index + local_i
            if (processed % ZS_CHECKPOINT_EVERY == 0) or (processed == total):
                _save_ckpt(processed, pred_ids_list, pred_labels, pred_rows)
                elapsed = time.time() - t0
                done_now = max(1, processed - start_index)
                eta_min = ((total - processed) * (elapsed / done_now)) / 60.0
                print(f"{model_name} progress: {processed}/{total} | checkpoint saved | ETA ~ {eta_min:.1f} min")

    if len(pred_labels) != total or len(pred_ids_list) != total:
        raise RuntimeError(
            f"Predictions incomplete for {model_name}: expected {total}, got labels={len(pred_labels)}, ids={len(pred_ids_list)}"
        )

    pred_ids = np.asarray(pred_ids_list, dtype=int)
    pred_df = pd.DataFrame(pred_rows)
    pred_df["is_match"] = pred_df["y_test_label"] == pred_df["nli_prediction"]

    joblib.dump(
        {
            "nli_pred_ids": pred_ids,
            "nli_pred_labels": pred_labels,
            "nli_pred_df": pred_df,
            "model": model_name,
            "candidate_labels": candidate_labels,
            "x_eval_size": total,
            "max_zs_cases": MAX_ZS_CASES,
        },
        predictions_path,
    )

    return pred_ids, pred_df, checkpoint_path, predictions_path

# ---------------------------------------------------------
# Run all 3 models and summarize
# ---------------------------------------------------------
multi_model_rows = []
multi_model_pred_store = {}

for model_name in ZS_MODELS:
    print("\n" + "=" * 100)
    print(f"Running zero-shot model: {model_name}")
    try:
        pred_ids, pred_df, ckpt_path, preds_path = _run_zeroshot_model(model_name)

        p, r, f1, _ = precision_recall_fscore_support(
            y_eval_idx,
            pred_ids,
            average="weighted",
            zero_division=0,
        )
        acc = accuracy_score(y_eval_idx, pred_ids)

        multi_model_rows.append({
            "model": model_name,
            "status": "ok",
            "accuracy": acc,
            "precision_weighted": p,
            "recall_weighted": r,
            "f1_weighted": f1,
            "support": int(len(y_eval_idx)),
            "checkpoint": str(ckpt_path),
            "predictions_artifact": str(preds_path),
            "error": "",
        })

        multi_model_pred_store[model_name] = {
            "pred_ids": pred_ids,
            "pred_df": pred_df,
            "checkpoint": str(ckpt_path),
            "predictions_artifact": str(preds_path),
        }

        print(f"Completed: {model_name}")
        print(f"Accuracy: {acc:.4f} | Precision: {p:.4f} | Recall: {r:.4f} | F1: {f1:.4f}")

    except Exception as e:
        multi_model_rows.append({
            "model": model_name,
            "status": "failed",
            "accuracy": np.nan,
            "precision_weighted": np.nan,
            "recall_weighted": np.nan,
            "f1_weighted": np.nan,
            "support": len(y_eval_idx),
            "checkpoint": "",
            "predictions_artifact": "",
            "error": str(e),
        })
        print(f"Failed: {model_name}")
        print(f"Error: {e}")

multi_model_metrics_df = pd.DataFrame(multi_model_rows)
display(multi_model_metrics_df)

# Save combined summary artifact
multi_summary_path = artifacts_dir_multi / f"nli_multi_model_summary_{total}.pkl"
joblib.dump(
    {
        "metrics_df": multi_model_metrics_df,
        "models": ZS_MODELS,
        "pred_store": multi_model_pred_store,
        "candidate_labels": candidate_labels,
        "x_eval_size": total,
        "max_zs_cases": MAX_ZS_CASES,
    },
    multi_summary_path,
 )
print(f"Saved multi-model summary: {multi_summary_path}")

Multi-model zero-shot device: cuda
Running multi-model zero-shot on 11923 tweets (MAX_ZS_CASES=None)

Running zero-shot model: facebook/bart-large-mnli


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

MoritzLaurer/deberta-v3-large-zeroshot-v2.0 progress: 1000/11923 | checkpoint saved | ETA ~ 4.5 min
MoritzLaurer/deberta-v3-large-zeroshot-v2.0 progress: 2000/11923 | checkpoint saved | ETA ~ 4.0 min
MoritzLaurer/deberta-v3-large-zeroshot-v2.0 progress: 3000/11923 | checkpoint saved | ETA ~ 3.6 min
MoritzLaurer/deberta-v3-large-zeroshot-v2.0 progress: 4000/11923 | checkpoint saved | ETA ~ 3.1 min
MoritzLaurer/deberta-v3-large-zeroshot-v2.0 progress: 5000/11923 | checkpoint saved | ETA ~ 2.7 min
MoritzLaurer/deberta-v3-large-zeroshot-v2.0 progress: 6000/11923 | checkpoint saved | ETA ~ 2.3 min
MoritzLaurer/deberta-v3-large-zeroshot-v2.0 progress: 7000/11923 | checkpoint saved | ETA ~ 1.9 min
MoritzLaurer/deberta-v3-large-zeroshot-v2.0 progress: 8000/11923 | checkpoint saved | ETA ~ 1.5 min
MoritzLaurer/deberta-v3-large-zeroshot-v2.0 progress: 9000/11923 | checkpoint saved | ETA ~ 1.2 min
MoritzLaurer/deberta-v3-large-zeroshot-v2.0 progress: 10000/11923 | checkpoint saved | ETA ~ 0.8 min

,model,status,accuracy,precision_weighted,recall_weighted,f1_weighted,support,checkpoint,predictions_artifact,error
0,facebook/bart-large-mnli,ok,0.446532,0.450896,0.446532,0.399407,11923,Artifacts/nli_checkpoint_facebook_bart-large-m...,Artifacts/nli_predictions_facebook_bart-large-...,
1,MoritzLaurer/ModernBERT-large-zeroshot-v2.0,ok,0.570410,0.631770,0.570410,0.550866,11923,Artifacts/nli_checkpoint_MoritzLaurer_ModernBE...,Artifacts/nli_predictions_MoritzLaurer_ModernB...,
2,MoritzLaurer/deberta-v3-large-zeroshot-v2.0,ok,0.554391,0.594193,0.554391,0.544434,11923,Artifacts/nli_checkpoint_MoritzLaurer_deberta-...,Artifacts/nli_predictions_MoritzLaurer_deberta...,


Saved multi-model summary: Artifacts/nli_multi_model_summary_11923.pkl


#### **Evaluation**
Print weighted evaluation metrics and full classification report for each successfully-run zero-shot model.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

if "multi_model_pred_store" not in globals():
    raise ValueError("multi_model_pred_store not found. Run the multi-model predictions cell first.")

if "y_eval_idx" not in globals() or "class_names_list" not in globals():
    raise ValueError("y_eval_idx/class_names_list not found. Run the zero-shot cells above first.")

eval_rows = []

for model_name, payload in multi_model_pred_store.items():
    pred_ids = np.asarray(payload.get("pred_ids", []), dtype=int)
    if len(pred_ids) != len(y_eval_idx):
        print(f"Skipping {model_name}: length mismatch ({len(pred_ids)} vs {len(y_eval_idx)})")
        continue

    p, r, f1, _ = precision_recall_fscore_support(
        y_eval_idx, pred_ids, average="weighted", zero_division=0
    )
    acc = accuracy_score(y_eval_idx, pred_ids)

    eval_rows.append({
        "Model": model_name,
        "Accuracy": acc,
        "Precision": p,
        "Recall": r,
        "F1-Score": f1,
        "Support": int(len(y_eval_idx)),
    })

eval_df = pd.DataFrame(eval_rows)
if not eval_df.empty:
    display(eval_df.set_index("Model"))
else:
    print("No successful model predictions available to evaluate.")

for model_name, payload in multi_model_pred_store.items():
    pred_ids = np.asarray(payload.get("pred_ids", []), dtype=int)
    if len(pred_ids) != len(y_eval_idx):
        continue

    print("\n" + "=" * 100)
    print(f"Classification report for: {model_name}")
    print(
        classification_report(
            y_eval_idx,
            pred_ids,
            target_names=class_names_list,
            zero_division=0,
        )
    )

,Accuracy,Precision,Recall,F1-Score,Support
Model,,,,,
facebook/bart-large-mnli,0.446532,0.450896,0.446532,0.399407,11923
MoritzLaurer/ModernBERT-large-zeroshot-v2.0,0.570410,0.631770,0.570410,0.550866,11923
MoritzLaurer/deberta-v3-large-zeroshot-v2.0,0.554391,0.594193,0.554391,0.544434,11923



Classification report for: facebook/bart-large-mnli
                     precision    recall  f1-score   support

                age       0.35      0.61      0.44      1952
          ethnicity       0.52      0.83      0.64      1995
             gender       0.38      0.63      0.48      1952
  not_cyberbullying       0.27      0.08      0.12      2040
other_cyberbullying       0.34      0.07      0.11      2006
           religion       0.86      0.49      0.62      1978

           accuracy                           0.45     11923
          macro avg       0.45      0.45      0.40     11923
       weighted avg       0.45      0.45      0.40     11923


Classification report for: MoritzLaurer/ModernBERT-large-zeroshot-v2.0
                     precision    recall  f1-score   support

                age       0.79      0.23      0.35      1952
          ethnicity       0.77      0.81      0.79      1995
             gender       0.50      0.76      0.60      1952
  not_cyberbullyi

### **OpenAI**

In [31]:
import os
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from openai import OpenAI

# ----------------------------
# OpenAI configuration
# ----------------------------
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
MAX_OPENAI_CASES = None  # set integer (e.g., 200 or 1000) for a smaller quick run
OPENAI_REQUEST_TIMEOUT_SECONDS = 60
OPENAI_MAX_RETRIES = 3

# Checkpoint/resume configuration (same style as Ollama)
OPENAI_CHECKPOINT_EVERY = 25
OPENAI_RESUME_FROM_CHECKPOINT = True
OPENAI_RESET_CHECKPOINT = False

# ----------------------------
# Load .env if present (without requiring python-dotenv)
# ----------------------------
def _load_env_file(env_path: str = ".env") -> None:
    if not os.path.exists(env_path):
        return
    try:
        with open(env_path, "r", encoding="utf-8") as f:
            for raw_line in f:
                line = raw_line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                if key and key not in os.environ:
                    os.environ[key] = value
    except Exception as e:
        print(f"Warning: Failed to read .env file: {e}")

_load_env_file()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY is not set. Put it in .env as OPENAI_API_KEY=... or export it in your environment."
    )

client = OpenAI(api_key=api_key, timeout=OPENAI_REQUEST_TIMEOUT_SECONDS)

# Reuse same eval split as Ollama section when available.
if "x_test" not in globals():
    x_test = joblib.load(r".\Artifacts\x_test.pkl")
if "y_test" not in globals():
    y_test = joblib.load(r".\Artifacts\y_test.pkl")
if "class_names" not in globals():
    class_names = joblib.load(r".\Artifacts\class_names.pkl")

class_names_list = list(class_names)
label_to_id = {label: i for i, label in enumerate(class_names_list)}
FALLBACK_LABEL = "other_cyberbullying"

if MAX_OPENAI_CASES is None:
    x_eval_openai = list(x_test)
    y_eval_openai = np.asarray(y_test)
else:
    x_eval_openai = list(x_test)[:MAX_OPENAI_CASES]
    y_eval_openai = np.asarray(y_test)[:MAX_OPENAI_CASES]

artifacts_dir = Path(r".\Artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
safe_openai_model = re.sub(r"[^a-zA-Z0-9_.-]", "_", OPENAI_MODEL)
openai_checkpoint_path = artifacts_dir / f"openai_checkpoint_{safe_openai_model}.pkl"
openai_predictions_path = artifacts_dir / f"openai_predictions_{safe_openai_model}.pkl"

if OPENAI_RESET_CHECKPOINT and openai_checkpoint_path.exists():
    openai_checkpoint_path.unlink()
    print(f"Deleted old OpenAI checkpoint: {openai_checkpoint_path}")

SYSTEM_PROMPT = f"""You are an expert annotator for cyberbullying tweet classification.

Valid labels: {class_names_list}

Rules:
1) Return exactly one label from the valid label list.
2) Output STRICT JSON only, with this schema:
   {{"label": "<one_valid_label>"}}
3) Do not output markdown, code fences, or any extra keys.
"""

def _parse_json_payload(raw_text: str):
    text = (raw_text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.IGNORECASE | re.DOTALL).strip()
    return json.loads(text)

def _normalize_text(s: str) -> str:
    return re.sub(r"[^a-z0-9_]+", "", (s or "").strip().lower())

normalized_label_to_canonical = {_normalize_text(label): label for label in class_names_list}

def _map_to_valid_label(raw_label: str) -> str:
    if raw_label in label_to_id:
        return raw_label

    raw_norm = _normalize_text(raw_label)
    if raw_norm in normalized_label_to_canonical:
        return normalized_label_to_canonical[raw_norm]

    for cls in class_names_list:
        cls_norm = _normalize_text(cls)
        if cls_norm and cls_norm in raw_norm:
            return cls

    if FALLBACK_LABEL in label_to_id:
        return FALLBACK_LABEL

    raise ValueError(
        f"Invalid label returned and fallback label not found: {raw_label}. "
        f"Expected one of {class_names_list} with fallback '{FALLBACK_LABEL}'."
    )

def openai_predict_one(tweet_text: str):
    last_error = None
    user_prompt = f"Classify this tweet:\n\n{tweet_text}"

    for attempt in range(1, OPENAI_MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
            )
            content = resp.choices[0].message.content
            payload = _parse_json_payload(content)
            raw_label = str(payload.get("label", "")).strip()
            final_label = _map_to_valid_label(raw_label)
            return label_to_id[final_label], final_label
        except Exception as err:
            last_error = err
            if attempt < OPENAI_MAX_RETRIES:
                time.sleep(1.2 * attempt)

    raise RuntimeError(f"OpenAI prediction failed after retries: {last_error}")

def save_openai_checkpoint(next_index: int, pred_ids, pred_rows):
    state = {
        "model": OPENAI_MODEL,
        "x_eval_size": len(x_eval_openai),
        "next_index": int(next_index),
        "pred_ids": list(pred_ids),
        "pred_rows": list(pred_rows),
        "saved_at_unix": time.time(),
    }
    joblib.dump(state, openai_checkpoint_path)

# ----------------------------
# Resume from checkpoint if available
# ----------------------------
openai_pred_ids = []
openai_rows = []
openai_start_index = 0

if OPENAI_RESUME_FROM_CHECKPOINT and openai_checkpoint_path.exists():
    state = joblib.load(openai_checkpoint_path)
    same_model = state.get("model") == OPENAI_MODEL
    same_size = int(state.get("x_eval_size", -1)) == len(x_eval_openai)
    if same_model and same_size:
        openai_pred_ids = list(state.get("pred_ids", []))
        openai_rows = list(state.get("pred_rows", []))
        openai_start_index = int(state.get("next_index", len(openai_pred_ids)))
        openai_start_index = max(0, min(openai_start_index, len(x_eval_openai)))
        print(f"Resumed OpenAI checkpoint: {openai_checkpoint_path}")
        print(f"Loaded {len(openai_pred_ids)} saved predictions; continuing from index {openai_start_index}")
    else:
        print("OpenAI checkpoint found but skipped due to model/test-size mismatch.")

# ----------------------------
# Run OpenAI prediction for each tweet
# ----------------------------
openai_total = len(x_eval_openai)
openai_start_time = time.time()

for idx in range(openai_start_index, openai_total):
    tweet_text = x_eval_openai[idx]
    pred_id, pred_label = openai_predict_one(str(tweet_text))
    openai_pred_ids.append(int(pred_id))
    openai_rows.append({
        "tweet": str(tweet_text),
        "y_test_label": class_names_list[int(y_eval_openai[idx])],
        "openai_prediction": pred_label,
    })

    processed = idx + 1
    if (processed % OPENAI_CHECKPOINT_EVERY == 0) or (processed == openai_total):
        save_openai_checkpoint(next_index=processed, pred_ids=openai_pred_ids, pred_rows=openai_rows)
        elapsed = time.time() - openai_start_time
        done_now = max(1, processed - openai_start_index)
        sec_per_tweet = elapsed / done_now
        remaining = openai_total - processed
        eta_min = (remaining * sec_per_tweet) / 60.0
        print(f"OpenAI progress: {processed}/{openai_total} | checkpoint saved | ETA ~ {eta_min:.1f} min")

# Convert and save final artifacts
openai_pred_ids = np.asarray(openai_pred_ids, dtype=int)
openai_pred_df = pd.DataFrame(openai_rows)
openai_pred_df["is_match"] = openai_pred_df["y_test_label"] == openai_pred_df["openai_prediction"]

openai_prediction_compare_df = openai_pred_df[[
    "tweet",
    "y_test_label",
    "openai_prediction",
    "is_match",
]].copy()

joblib.dump(
    {
        "openai_pred_ids": openai_pred_ids,
        "prediction_compare_df": openai_prediction_compare_df,
        "model": OPENAI_MODEL,
        "x_eval_size": len(x_eval_openai),
    },
    openai_predictions_path,
 )

display(openai_prediction_compare_df.head(20))
print(f"Done. Collected {len(openai_pred_ids)} predictions from model: {OPENAI_MODEL}")
print(f"Total exact label matches: {int(openai_prediction_compare_df['is_match'].sum())}/{len(openai_prediction_compare_df)}")
print(f"OpenAI checkpoint file: {openai_checkpoint_path}")
print(f"OpenAI predictions artifact: {openai_predictions_path}")

OpenAI progress: 25/11923 | checkpoint saved | ETA ~ 175.0 min
OpenAI progress: 50/11923 | checkpoint saved | ETA ~ 158.4 min
OpenAI progress: 75/11923 | checkpoint saved | ETA ~ 155.9 min
OpenAI progress: 100/11923 | checkpoint saved | ETA ~ 152.8 min
OpenAI progress: 125/11923 | checkpoint saved | ETA ~ 146.7 min
OpenAI progress: 150/11923 | checkpoint saved | ETA ~ 148.2 min
OpenAI progress: 175/11923 | checkpoint saved | ETA ~ 151.1 min
OpenAI progress: 200/11923 | checkpoint saved | ETA ~ 152.5 min
OpenAI progress: 225/11923 | checkpoint saved | ETA ~ 157.9 min
OpenAI progress: 250/11923 | checkpoint saved | ETA ~ 158.9 min
OpenAI progress: 275/11923 | checkpoint saved | ETA ~ 156.9 min
OpenAI progress: 300/11923 | checkpoint saved | ETA ~ 157.2 min
OpenAI progress: 325/11923 | checkpoint saved | ETA ~ 154.9 min
OpenAI progress: 350/11923 | checkpoint saved | ETA ~ 153.5 min
OpenAI progress: 375/11923 | checkpoint saved | ETA ~ 150.3 min
OpenAI progress: 400/11923 | checkpoint sav

,tweet,y_test_label,openai_prediction,is_match
0,today tomorrow try move mountains cross fingers,other_cyberbullying,not_cyberbullying,False
1,typa girl bullied lesbians high school,age,gender,False
2,cool woman gave directions polling place whose...,age,gender,False
3,need teach version quran terrorists killing pe...,religion,religion,True
4,rome destroy coliseum roman forum every buildi...,religion,religion,True
5,try hide jihadi propaganda know hell gys specl...,religion,religion,True
6,city college,not_cyberbullying,not_cyberbullying,True
7,pepperidge farm remembers,other_cyberbullying,not_cyberbullying,False
8,added feminazi white knight professional victi...,other_cyberbullying,gender,False
9,blm something know muslim leaders record stati...,religion,religion,True


Done. Collected 11923 predictions from model: gpt-4.1-mini
Total exact label matches: 6215/11923
OpenAI checkpoint file: Artifacts\openai_checkpoint_gpt-4.1-mini.pkl
OpenAI predictions artifact: Artifacts\openai_predictions_gpt-4.1-mini.pkl


#### **Evaluation**

In [32]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# Ensure y_eval is integer index array and lengths match OpenAI predictions.
y_eval_openai_idx = np.asarray(y_eval_openai).astype(int)
openai_pred_idx = np.asarray(openai_pred_ids).astype(int)

if len(y_eval_openai_idx) != len(openai_pred_idx):
    raise ValueError(
        f"Length mismatch: y_eval_openai has {len(y_eval_openai_idx)} rows, openai_pred has {len(openai_pred_idx)} rows"
    )

# Keep readable labels and index columns for inspection.
openai_pred_df["true_label_idx"] = y_eval_openai_idx
openai_pred_df["openai_pred_idx"] = openai_pred_idx
display(openai_pred_df.head(10))

p, r, f1, support = precision_recall_fscore_support(
    y_eval_openai_idx,
    openai_pred_idx,
    average="weighted",
    zero_division=0,
 )

openai_metrics = {
    "Accuracy": accuracy_score(y_eval_openai_idx, openai_pred_idx),
    "Precision": p,
    "Recall": r,
    "F1-Score": f1,
    "Support": int(len(y_eval_openai_idx)),
}

openai_metrics_df = pd.DataFrame(openai_metrics, index=[OPENAI_MODEL]).T
display(openai_metrics_df)

print("\nOpenAI classification report:\n")
print(
    classification_report(
        y_eval_openai_idx,
        openai_pred_idx,
        target_names=class_names_list,
        zero_division=0,
    )
)

,tweet,y_test_label,openai_prediction,is_match,true_label_idx,openai_pred_idx
0,today tomorrow try move mountains cross fingers,other_cyberbullying,not_cyberbullying,False,4,3
1,typa girl bullied lesbians high school,age,gender,False,0,2
2,cool woman gave directions polling place whose...,age,gender,False,0,2
3,need teach version quran terrorists killing pe...,religion,religion,True,5,5
4,rome destroy coliseum roman forum every buildi...,religion,religion,True,5,5
5,try hide jihadi propaganda know hell gys specl...,religion,religion,True,5,5
6,city college,not_cyberbullying,not_cyberbullying,True,3,3
7,pepperidge farm remembers,other_cyberbullying,not_cyberbullying,False,4,3
8,added feminazi white knight professional victi...,other_cyberbullying,gender,False,4,2
9,blm something know muslim leaders record stati...,religion,religion,True,5,5


,gpt-4.1-mini
Accuracy,0.521261
Precision,0.599045
Recall,0.521261
F1-Score,0.498640
Support,11923.000000



OpenAI classification report:

                     precision    recall  f1-score   support

                age       0.71      0.04      0.08      1952
          ethnicity       0.85      0.87      0.86      1995
             gender       0.58      0.42      0.49      1952
  not_cyberbullying       0.40      0.77      0.53      2040
other_cyberbullying       0.14      0.19      0.16      2006
           religion       0.92      0.82      0.87      1978

           accuracy                           0.52     11923
          macro avg       0.60      0.52      0.50     11923
       weighted avg       0.60      0.52      0.50     11923

